In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

import os 
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
data = data.drop(columns=['EmployeeCount', 'StandardHours', 'EmployeeNumber', 'Over18'])

print(data.shape)
data.head(10)

In [ ]:
# Encode target first, separately
target = 'Attrition'

# Separate feature types
categorical_cols = data.select_dtypes(include='object').columns.tolist()
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from categorical list
categorical_cols.remove(target)

print("Categorical columns:", categorical_cols)
print("\nNumerical columns:", numerical_cols)
print(f"\nTotal: {len(categorical_cols)} categorical, {len(numerical_cols)} numerical")

### Understanding Columns

- Binary Categories: Gender, OverTime
- Nominal Categories: BusinessTravel, Department, EducationField, JobRole, MaritalStatus
- Numerical Ordinal Categories: Education, EnvironmentSatisfaction, JobInvolvement, JobLevel, JobSatisfaction, PerformanceRating, RelationshipSatisfaction, StockOptionLevel, WorkLifeBalance

In [ ]:
le_target = LabelEncoder()
data['Attrition'] = le_target.fit_transform(data['Attrition'])

print('Target Encoding: ', dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))
print("Value counts:\n", data['Attrition'].value_counts())

In [ ]:
binary_cols = ['Gender', 'OverTime']

le = LabelEncoder()

for col in binary_cols:
    data[col] = le.fit_transform(data[col])
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
nominal_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']

data = pd.get_dummies(data, columns=nominal_cols, drop_first=True)

print("Shape after one-hot encoding:", data.shape)
print("New columns added:", [c for c in data.columns if any(nom in c for nom in nominal_cols)])

In [ ]:
print('Nulls: ', data.isnull().sum().sum())
print('Remaining Objects: ', data.select_dtypes(include='object').columns.tolist())
print('\nData Types Summary: ')
print(data.dtypes.value_counts())
data.head(3)

In [ ]:
x = data.drop(columns=['Attrition'])
y = data['Attrition']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("Training set:", x_train.shape, "| Attrition rate:", round(y_train.mean() * 100, 2), "%")
print("Test set:", x_test.shape, "| Attrition rate:", round(y_test.mean() * 100, 2), "%")

In [ ]:
# Columns to scale — all numerical except one-hot encoded cols (already 0/1)
# We scale everything that isn't binary (0/1 already)
cols_to_scale = ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

scaler = StandardScaler()

# Fit ONLY on training data, then transform both
x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler.transform(x_test[cols_to_scale])  # <-- transform only, no fit

print("Scaling complete.")
print("MonthlyIncome mean after scaling (train):", round(x_train['MonthlyIncome'].mean(), 4))
print("MonthlyIncome std after scaling (train):", round(x_train['MonthlyIncome'].std(), 4))

In [ ]:
print("Before SMOTE:")
print(f"  Training samples: {x_train.shape[0]}")
print(f"  Class distribution: {dict(pd.Series(y_train).value_counts().sort_index())}")

smote = SMOTE(random_state=42)
x_train_sm, y_train_sm = smote.fit_resample(x_train, y_train)

print("\nAfter SMOTE:")
print(f"  Training samples: {x_train_sm.shape[0]}")
print(f"  Class distribution: {dict(pd.Series(y_train_sm).value_counts().sort_index())}")
print(f"  Attrition rate: {round(y_train_sm.mean() * 100, 2)}%")

print("\nTest set unchanged:")
print(f"  Test samples: {x_test.shape[0]}")
print(f"  Class distribution: {dict(pd.Series(y_test).value_counts().sort_index())}")

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)

# Save processed dataframes
x_train_sm.to_csv('../data/processed/x_train.csv', index=False)
x_test.to_csv('../data/processed/x_test.csv', index=False)
pd.Series(y_train_sm, name='Attrition').to_csv('../data/processed/y_train.csv', index=False)
pd.Series(y_test, name='Attrition').to_csv('../data/processed/y_test.csv', index=False)

# Save the scaler — critical for production use
with open('../outputs/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Saved:")
print("  data/processed/x_train.csv")
print("  data/processed/x_test.csv")
print("  data/processed/y_train.csv")
print("  data/processed/y_test.csv")
print("  outputs/models/scaler.pkl")

In [ ]:
print("=" * 62)
print("       PREPROCESSING SUMMARY — 02_preprocessing.ipynb")
print("=" * 62)

print("""
ENCODING
  Binary (Label Encoded):   Gender, OverTime
  Nominal (One-Hot):        BusinessTravel, Department, EducationField, JobRole, MaritalStatus
  Target:                   Attrition → No=0, Yes=1

SHAPE
  Raw input:                (1470, 31)
  After encoding:           (1470, 45)
  After split:              Train (1176, 44) | Test (294, 44)

SCALING
  Method:                   StandardScaler (mean=0, std=1)
  Fitted on:                Training data only
  Applied to:               23 numerical columns

SMOTE
  Applied to:               Training data only
  Before:                   986 No | 190 Yes (1176 total)
  After:                    986 No | 986 Yes (1972 total)
  Test set:                 247 No | 47 Yes  (294 total, unchanged)

SAVED Training and Testing Sets and Scaler
""")
print("=" * 33)
print("  Ready for: 03_modeling.ipynb")
print("=" * 33)